Circadian cycle analysis using unfiltered PLV features.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import matplotlib.transforms as mpltransforms

import seaborn as sns
import os
import datetime
import pickle

from astropy.timeseries import LombScargle
from astropy.stats.circstats import circmean, circvar
from scipy.signal import hilbert

In [ ]:
if os.getcwd().endswith('/notebooks/figures'):
    os.chdir('../../')

In [ ]:
from src.dataproc import spectralanalysis
from src.dataproc import utils

## Supporting function

## LS fit to ECoG features

In [ ]:
FEATURE_TYPE = 'plv'  # 'plv'
DATA_RANGE = 'interval'  # 'total' or 'interval'
drsuff_dict = {'total':'', 'interval':'_interval'}  # data range suffix to be added to folder name

implantinfo_path = 'data/RNSdata/Outcome/Hippo_outcomes_postprocessed_2025April08.csv'

if FEATURE_TYPE == 'plv':
    featdf_path = 'data/processed/ecogfeatures/plv_postproc/%s_trigger-Scheduled.csv'
    savedat_folder = f"data/processed/lscycles/plv_postproc{drsuff_dict[DATA_RANGE]}/"
    savefig_folder = f"output/notebooks/Compute_Circadian_Cycles_v02/plv_postproc{drsuff_dict[DATA_RANGE]}/"
    

In [ ]:
savedat_flag = False

if (savedat_flag) and (not os.path.exists(savedat_folder)): # if dat folder doesnt exist, then make it
    os.makedirs(savedat_folder)

In [ ]:
implantdf = pd.read_csv(implantinfo_path)   # date when RNS was implanted

patient_list = ['NPE%04d'%(ii+1) for ii in range(140)]

In [ ]:
# read the feature csv file and store in a dictionary

implant_info_patlist = implantdf['npe_code'].to_list()

featdf_dict = {}

for patient_id in patient_list[:]:
    
    if not os.path.exists(featdf_path % (patient_id)):
        print(f'{patient_id}: No feature file found. Removing from patient list.')
        featdf_dict[patient_id] = None
        continue
    
    print(f'{patient_id}: Exists.')
    
    # read data
    featdf = pd.read_csv(featdf_path % (patient_id))
    featdf['Round_local_hour'] = featdf['Round local timestamp'].apply(lambda x: datetime.datetime.fromisoformat(x).hour).values
    
    feature_list = [item for item in featdf.columns if ('Ch_' in item)]  # list of features
    
    # check if patient is available in the implant dataframe
    if patient_id in implant_info_patlist:
        implant_date = datetime.datetime.fromisoformat(implantdf.loc[implantdf['npe_code']==patient_id]['implant_date'].item())
        utils.tfromstart_days(featdf, start_date=implant_date)  # time from the date of implantation
        print(f'>> {patient_id}: Implant date used as start date.')
        
    else:  # implant info unavailable; use the recordings dataframe to pick the start of recordings
        utils.tfromstart_days(featdf)  # time from start of recordings in days
    
    patccsdf = spectralanalysis.lscircadian_different_durs(patient_id, featdf, feature_list, rangetype=DATA_RANGE)
    
    # store the result
    featdf_dict[patient_id] = featdf.copy()
    
    if savedat_flag:
        save_filename = '%s%s.csv'%(savedat_folder, patient_id)
        patccsdf.to_csv(save_filename, index=False)